# Faruq-v3 — AF2 × D-FINE-N seed-42 transfer screen

This notebook is intentionally thin. The auditable workflow now runs through `scripts/run_dfine_af2_kaggle_screen_v2.py`.

v2 fixes only the custom-dataset category indexing contract required by D-FINE when `remap_mscoco_category=False`: the 21 coffee classes are passed as target IDs `0..20`, not `1..21`. No scientific setting or promotion threshold is changed.

Comparison: `DFN0` vs `frozen AF2 + D-FINE-N`, validation only. Locked test remains closed.

Required Kaggle input: exactly one `faruq-development-v3-grouped.tar.bin`. Optional resume input: `af2-dfine-n-transfer-seed42-state.zip`. GPU and Internet must be enabled.


In [ ]:
from pathlib import Path
import shutil, subprocess, sys

WORK=Path('/kaggle/working')
if not WORK.is_dir():
    raise RuntimeError('Kaggle-only notebook')
COFFEE=WORK/'coffee-bean-detection'
DFINE=WORK/'D-FINE'
BRANCH='agent/af2-dfine-n-transfer-screen'
DFINE_COMMIT='956d1709314c2c6a4df6f34de232054578a7449f'

for path in (COFFEE,DFINE):
    if path.exists():
        shutil.rmtree(path)

subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(COFFEE)],check=True)
subprocess.run(['git','clone','--depth','1','https://github.com/Peterande/D-FINE.git',str(DFINE)],check=True)
head=subprocess.check_output(['git','rev-parse','HEAD'],cwd=DFINE,text=True).strip()
if head!=DFINE_COMMIT:
    subprocess.run(['git','fetch','origin',DFINE_COMMIT,'--depth','1'],cwd=DFINE,check=True)
    subprocess.run(['git','checkout','--detach',DFINE_COMMIT],cwd=DFINE,check=True)
head=subprocess.check_output(['git','rev-parse','HEAD'],cwd=DFINE,text=True).strip()
if head!=DFINE_COMMIT:
    raise RuntimeError(f'D-FINE commit mismatch: {head}')

subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(COFFEE)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','faster-coco-eval>=1.6.6','tensorboard','scipy','calflops','transformers','loguru','PyYAML'],check=True)
print('COFFEE:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=COFFEE,text=True).strip())
print('D-FINE:',head)


In [ ]:
subprocess.run([sys.executable,'-u',str(COFFEE/'scripts/run_dfine_af2_kaggle_screen_v2.py')],cwd=COFFEE,check=True)
